In [4]:
%cd /home/brimmann/works/xRAG

/home/brimmann/works/xRAG


/home/brimmann/works/xRAG/.venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [7]:
from huggingface_hub import notebook_login

notebook_login()

In [5]:
import torch
from transformers import AutoTokenizer
from src.distill.modeling_xgemma import XGemmaForCausalLM, XGemmaConfig

# Specify model and device
model_name = "google/gemma-2-2b"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [8]:
# Load Tokenizer and add the special XRAG token
tokenizer = AutoTokenizer.from_pretrained(model_name)
xrag_token = "<xRAG>"
tokenizer.add_special_tokens({"additional_special_tokens": [xrag_token]})
xrag_token_id = tokenizer.convert_tokens_to_ids(xrag_token)

print(f"XRAG token '{xrag_token}' added with ID: {xrag_token_id}")

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

XRAG token '<xRAG>' added with ID: 256000


In [10]:
# Create a config with our custom parameters
retriever_hidden_size = 128
config = XGemmaConfig.from_pretrained(
    model_name,
    projector_type='mlp2x_gelu',
    retriever_hidden_size=retriever_hidden_size,
)

# Load the model with this config
model = XGemmaForCausalLM.from_pretrained(model_name, config=config)

# Resize token embeddings layer to account for the new special token
model.resize_token_embeddings(len(tokenizer))

# Set the special token ID on the model instance and move to device
model.set_xrag_token_id(xrag_token_id)
model.to(device)
model.eval()

print(f"Model {model_name} loaded successfully onto {device}.")
print(f"Projector is configured: {hasattr(model, 'projector')}")

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:02<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/481M [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of XGemmaForCausalLM were not initialized from the model checkpoint at google/gemma-2-2b and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model google/gemma-2-2b loaded successfully onto cuda.
Projector is configured: True


In [11]:
# Prepare dummy inputs
prompt = f"The following document provides information about Gemini. Document: {xrag_token}. Based on this document, Gemini is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

# Count xrag tokens to create corresponding dummy retrieval embeddings
num_xrag_tokens = torch.sum(input_ids == xrag_token_id).item()
print(f"Found {num_xrag_tokens} XRAG token(s) in the prompt.")

# Create random retrieval embeddings
dummy_retrieval_embeds = torch.randn(num_xrag_tokens, retriever_hidden_size).to(device)

Found 1 XRAG token(s) in the prompt.


In [12]:
print("--- Testing generation WITH retrieval embeddings ---")
try:
    with torch.no_grad():
        output_ids_with_retrieval = model.generate(
            input_ids=input_ids,
            retrieval_embeds=dummy_retrieval_embeds,
            max_new_tokens=20,
            do_sample=False
        )
    decoded_output = tokenizer.decode(output_ids_with_retrieval[0], skip_special_tokens=False)
    print("Generation successful.")
    print("Generated output:")
    print(decoded_output)
except Exception as e:
    print(f"Generation FAILED with an error: {e}")

--- Testing generation WITH retrieval embeddings ---
Generation successful.
Generated output:
 a cryptocurrency that is based on the Bitcoin blockchain.

Gemini is a cryptocurrency exchange that allows users to


In [13]:
print("\n--- Testing generation WITHOUT retrieval embeddings ---")
plain_prompt = "The capital of France is"
input_ids_plain = tokenizer(plain_prompt, return_tensors="pt").input_ids.to(device)

try:
    with torch.no_grad():
        output_ids_without_retrieval = model.generate(
            input_ids=input_ids_plain,
            max_new_tokens=5,
            do_sample=False
        )
    decoded_output_plain = tokenizer.decode(output_ids_without_retrieval[0], skip_special_tokens=True)
    print("Generation successful.")
    print("Generated output:")
    print(f"'{decoded_output_plain}'")
except Exception as e:
    print(f"Generation FAILED with an error: {e}")


--- Testing generation WITHOUT retrieval embeddings ---
Generation successful.
Generated output:
'The capital of France is a city that is full'
